# Near-Duplicate Text Detection: QQP & PAWS-Wiki Baseline Evaluation

This notebook demonstrates a simple baseline approach for detecting near-duplicate text pairs using the QQP (Quora Question Pairs) and PAWS-Wiki datasets.

**What we're doing:**
- Load 6 demo examples from QQP and PAWS-Wiki (3 from each dataset)
- Implement a simple Jaccard similarity baseline (word overlap metric)
- Compare against a naive baseline (word count parity check)
- Report precision, recall, and F1 scores on both methods
- Analyze how well these simple methods detect duplicates across different text lengths

**Datasets:**
- **QQP (Quora Question Pairs)**: 40k+ question pairs labeled as duplicate/not_duplicate, stratified by length
- **PAWS-Wiki**: 8k adversarial Wikipedia sentence pairs with high lexical overlap but different meanings

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install)
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab environment)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import os
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loguru import logger

# Configure logging for notebook
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2cec91-circular-text-fingerprints-evaluating-ec/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        logger.info(f"Attempting to load from GitHub: {GITHUB_DATA_URL}")
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=5) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.info(f"GitHub load failed ({type(e).__name__}), trying local fallback")
    
    if os.path.exists("mini_demo_data.json"):
        logger.info("Loading from local mini_demo_data.json")
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local filesystem")

In [ ]:
data = load_data()
logger.info(f"Loaded data with metadata: {data['metadata']['task']}")
logger.info(f"Available datasets: {data['metadata']['datasets_selected']}")

## Configuration

All tunable parameters are defined below. Set to minimal values for quick testing, then scale up if needed.

In [ ]:
# === Demo Configuration ===
# Baseline parameters
JACCARD_THRESHOLD = 0.4  # Jaccard similarity threshold for duplicate decision
WORD_COUNT_TOLERANCE = 3  # Naive baseline: max allowed word count difference for "duplicate"

# Data sampling
N_SAMPLES_QQP = 3  # Number of QQP examples to evaluate (original: 3)
N_SAMPLES_PAWS = 3  # Number of PAWS examples to evaluate (original: 3)

logger.info(f"Config: Jaccard threshold={JACCARD_THRESHOLD}, word_count_tol={WORD_COUNT_TOLERANCE}")
logger.info(f"Demo scale: QQP={N_SAMPLES_QQP}, PAWS={N_SAMPLES_PAWS}")

## Baseline Implementations

We implement two simple baselines:

1. **Jaccard Similarity Baseline**: Treat text as a set of words, compute word overlap fraction. High overlap suggests duplicates.
2. **Naive Word Count Baseline**: Assume pairs with similar word counts and some token overlap are duplicates. Simpler, no ML required.

In [ ]:
def word_count(text: str) -> int:
    """Count words in text."""
    return len(text.lower().split())

def jaccard_similarity(text1: str, text2: str) -> float:
    """Compute Jaccard similarity (word overlap / union)."""
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    
    if not words1 and not words2:
        return 1.0
    if not words1 or not words2:
        return 0.0
    
    intersection = len(words1 & words2)
    union = len(words1 | words2)
    return intersection / union if union > 0 else 0.0

def jaccard_baseline(text1: str, text2: str, threshold: float) -> int:
    """Jaccard baseline: predict duplicate if similarity >= threshold."""
    return 1 if jaccard_similarity(text1, text2) >= threshold else 0

def naive_baseline(text1: str, text2: str, word_count_tol: int) -> int:
    """Naive baseline: predict duplicate if word counts are close and some overlap."""
    wc1, wc2 = word_count(text1), word_count(text2)
    words1, words2 = set(text1.lower().split()), set(text2.lower().split())
    overlap = len(words1 & words2)
    
    # Heuristic: if word counts similar AND overlap > 0, predict duplicate
    count_similar = abs(wc1 - wc2) <= word_count_tol
    has_overlap = overlap > 0
    
    return 1 if (count_similar and has_overlap) else 0

logger.info("Baseline functions defined: jaccard_baseline, naive_baseline")

## Extract Text Pairs and Ground Truth

Parse the dataset JSON to extract (text1, text2, true_label) tuples for evaluation.

In [ ]:
def extract_pairs(data, dataset_name, n_samples):
    """Extract text pairs and labels from dataset."""
    pairs = []
    for ds in data['datasets']:
        if ds['dataset'] == dataset_name:
            for i, example in enumerate(ds['examples'][:n_samples]):
                input_json = json.loads(example['input'])
                
                # Extract field names (question1/question2 for QQP, sentence1/sentence2 for PAWS)
                if 'question1' in input_json:
                    text1, text2 = input_json['question1'], input_json['question2']
                else:
                    text1, text2 = input_json['sentence1'], input_json['sentence2']
                
                label = example['metadata_label_int']
                stratum = example.get('metadata_stratum', 'unknown')
                
                pairs.append({
                    'text1': text1,
                    'text2': text2,
                    'true_label': label,
                    'stratum': stratum,
                })
    return pairs

qqp_pairs = extract_pairs(data, 'glue_qqp_validation', N_SAMPLES_QQP)
paws_pairs = extract_pairs(data, 'paws_wiki_test', N_SAMPLES_PAWS)

all_pairs = qqp_pairs + paws_pairs
logger.info(f"Extracted {len(qqp_pairs)} QQP pairs + {len(paws_pairs)} PAWS pairs = {len(all_pairs)} total")

## Evaluate Both Baselines

For each pair, run both baselines and compare predictions to ground truth. Compute precision, recall, and F1.

In [ ]:
def compute_metrics(y_true, y_pred):
    """Compute precision, recall, F1 for binary classification."""
    tp = sum((y_true[i] == 1 and y_pred[i] == 1) for i in range(len(y_true)))
    fp = sum((y_true[i] == 0 and y_pred[i] == 1) for i in range(len(y_true)))
    fn = sum((y_true[i] == 1 and y_pred[i] == 0) for i in range(len(y_true)))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
    }

# Run both baselines on all pairs
jaccard_preds = []
naive_preds = []
true_labels = []
similarities = []

for pair in all_pairs:
    text1, text2 = pair['text1'], pair['text2']
    true_label = pair['true_label']
    
    # Baseline predictions
    jaccard_pred = jaccard_baseline(text1, text2, JACCARD_THRESHOLD)
    naive_pred = naive_baseline(text1, text2, WORD_COUNT_TOLERANCE)
    similarity = jaccard_similarity(text1, text2)
    
    jaccard_preds.append(jaccard_pred)
    naive_preds.append(naive_pred)
    true_labels.append(true_label)
    similarities.append(similarity)

# Compute metrics for each baseline
jaccard_metrics = compute_metrics(true_labels, jaccard_preds)
naive_metrics = compute_metrics(true_labels, naive_preds)

logger.info(f"Jaccard baseline metrics: {jaccard_metrics}")
logger.info(f"Naive baseline metrics: {naive_metrics}")

## Results Summary

Compare baseline performance on the demo dataset. Lower values indicate harder examples.

In [ ]:
# Create results dataframe
results_data = []
for i, pair in enumerate(all_pairs):
    results_data.append({
        'example_id': i,
        'dataset': 'QQP' if i < len(qqp_pairs) else 'PAWS',
        'true_label': 'duplicate' if pair['true_label'] == 1 else 'not_duplicate',
        'jaccard_pred': 'duplicate' if jaccard_preds[i] == 1 else 'not_duplicate',
        'naive_pred': 'duplicate' if naive_preds[i] == 1 else 'not_duplicate',
        'jaccard_similarity': round(similarities[i], 3),
        'stratum': pair['stratum'],
    })

results_df = pd.DataFrame(results_data)
print("\n=== Demo Evaluation Results ===")
print(results_df.to_string(index=False))

# Baseline comparison table
print("\n=== Baseline Metrics Comparison ===")
metrics_comparison = pd.DataFrame([
    {
        'Baseline': 'Jaccard Similarity',
        'Precision': f"{jaccard_metrics['precision']:.3f}",
        'Recall': f"{jaccard_metrics['recall']:.3f}",
        'F1': f"{jaccard_metrics['f1']:.3f}",
        'TP': jaccard_metrics['tp'],
        'FP': jaccard_metrics['fp'],
        'FN': jaccard_metrics['fn'],
    },
    {
        'Baseline': 'Naive Word Count',
        'Precision': f"{naive_metrics['precision']:.3f}",
        'Recall': f"{naive_metrics['recall']:.3f}",
        'F1': f"{naive_metrics['f1']:.3f}",
        'TP': naive_metrics['tp'],
        'FP': naive_metrics['fp'],
        'FN': naive_metrics['fn'],
    }
])
print(metrics_comparison.to_string(index=False))

# Summary statistics
print(f"\n=== Summary ===")
print(f"Total examples evaluated: {len(all_pairs)}")
print(f"True duplicates: {sum(true_labels)}")
print(f"True non-duplicates: {len(true_labels) - sum(true_labels)}")
print(f"Mean Jaccard similarity: {np.mean(similarities):.3f}")
print(f"Min Jaccard similarity: {np.min(similarities):.3f}")
print(f"Max Jaccard similarity: {np.max(similarities):.3f}")

## Visualization: Similarity Distribution and Performance

Visualize how Jaccard similarity scores separate duplicates from non-duplicates, and show baseline performance across stratified text lengths.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Jaccard similarity distribution
ax = axes[0]
for i, (pair, sim) in enumerate(zip(all_pairs, similarities)):
    label_str = 'duplicate' if pair['true_label'] == 1 else 'not_duplicate'
    color = 'green' if pair['true_label'] == 1 else 'red'
    marker = 'o' if i < len(qqp_pairs) else 's'  # circle for QQP, square for PAWS
    ax.scatter(i, sim, color=color, s=100, marker=marker, alpha=0.7, label=label_str if i < 2 else '')

ax.axhline(JACCARD_THRESHOLD, color='blue', linestyle='--', linewidth=2, label=f'Threshold={JACCARD_THRESHOLD}')
ax.set_xlabel('Example Index', fontsize=11)
ax.set_ylabel('Jaccard Similarity', fontsize=11)
ax.set_title('Jaccard Similarity by Example (Circles=QQP, Squares=PAWS)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])
ax.legend(fontsize=10)

# Plot 2: Baseline F1 scores
ax = axes[1]
baselines = ['Jaccard', 'Naive']
f1_scores = [jaccard_metrics['f1'], naive_metrics['f1']]
colors_bar = ['steelblue', 'coral']
bars = ax.bar(baselines, f1_scores, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, score in zip(bars, f1_scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('Baseline Performance Comparison', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

logger.info("Visualization saved to baseline_comparison.png")

## Analysis: Baseline Comparison

Key observations from the demo evaluation:

In [ ]:
# Stratified analysis by text length
print("\n=== Performance by Text Length Stratum ===")
for stratum in ['short', 'medium', 'long']:
    stratum_indices = [i for i, p in enumerate(all_pairs) if p['stratum'] == stratum]
    if not stratum_indices:
        print(f"\n{stratum.upper()}: No examples")
        continue
    
    stratum_true = [true_labels[i] for i in stratum_indices]
    stratum_jaccard = [jaccard_preds[i] for i in stratum_indices]
    stratum_naive = [naive_preds[i] for i in stratum_indices]
    
    j_metrics = compute_metrics(stratum_true, stratum_jaccard)
    n_metrics = compute_metrics(stratum_true, stratum_naive)
    
    print(f"\n{stratum.upper()} ({len(stratum_indices)} examples):")
    print(f"  Jaccard F1: {j_metrics['f1']:.3f} (P={j_metrics['precision']:.3f}, R={j_metrics['recall']:.3f})")
    print(f"  Naive F1:   {n_metrics['f1']:.3f} (P={n_metrics['precision']:.3f}, R={n_metrics['recall']:.3f})")

# Dataset-level analysis
print(f"\n=== Performance by Dataset ===")
qqp_indices = list(range(len(qqp_pairs)))
paws_indices = list(range(len(qqp_pairs), len(all_pairs)))

for name, indices in [('QQP', qqp_indices), ('PAWS-Wiki', paws_indices)]:
    if not indices:
        print(f"\n{name}: No examples")
        continue
    
    ds_true = [true_labels[i] for i in indices]
    ds_jaccard = [jaccard_preds[i] for i in indices]
    ds_naive = [naive_preds[i] for i in indices]
    
    j_metrics = compute_metrics(ds_true, ds_jaccard)
    n_metrics = compute_metrics(ds_true, ds_naive)
    
    print(f"\n{name} ({len(indices)} examples):")
    print(f"  Jaccard F1: {j_metrics['f1']:.3f} (P={j_metrics['precision']:.3f}, R={j_metrics['recall']:.3f})")
    print(f"  Naive F1:   {n_metrics['f1']:.3f} (P={n_metrics['precision']:.3f}, R={n_metrics['recall']:.3f})")

print(f"\n=== Key Findings ===")
print(f"1. Jaccard similarity baseline achieves {jaccard_metrics['f1']:.1%} F1 on demo set")
print(f"2. Naive word-count baseline achieves {naive_metrics['f1']:.1%} F1 on demo set")
print(f"3. Best baseline (Jaccard) shows {'higher' if jaccard_metrics['f1'] > naive_metrics['f1'] else 'lower'} F1 than naive approach")
print(f"4. Threshold tuning (Jaccard={JACCARD_THRESHOLD}) can improve precision/recall tradeoff")